# Today's Topic: Decorators

A **decorator** is a function that takes another function and extends or modifies its behavior without explicitly changing its source code.

**Why decorators?**
- **Code reuse** – common logic (logging, timing, access control) can be applied to many functions.
- **Separation of concerns** – keep core business logic clean and externalise cross‑cutting concerns.
- **Readability** – using `@decorator` syntax is clear and declarative.

**Prerequisites:** functions as first‑class objects, nested functions, closures.

We'll cover:
1. First‑class functions & closures (refresher)
2. Building a simple decorator
3. The `@` syntactic sugar
4. Decorators with arguments
5. Chaining multiple decorators
6. Real‑world use cases (logging, timing, authorisation)
7. Preserving function metadata with `functools.wraps`
8. Class‑based decorators

## 1. First‑Class Functions and Closures (Refresher)

In Python, functions are **first‑class objects**:
- can be assigned to variables
- can be passed as arguments
- can be returned from other functions

A **closure** is a nested function that remembers variables from its enclosing scope even after that scope has finished executing.

In [1]:
# First-class function example
def say_hello(name):
    return f"Hello, {name}!"

# Assign to a variable
greet = say_hello
print(greet("Alice"))

Hello, Alice!


In [2]:
# Pass as argument
def run_twice(func, arg):
    return func(arg), func(arg)

print(run_twice(say_hello, "Bob"))

('Hello, Bob!', 'Hello, Bob!')


In [3]:
# Closure example - closure means that after double variable receives the multiplier function double/multiplier still remembers the factor value which was available in the make_multiplier function which has already been terminated.
# It is the inner multiplier function—the one referenced by the double or triple variables—that actually remembers the factor. 
# When make_multiplier is called, it creates a scope containing the factor variable. 
# The inner function references that variable, and even after the outer function returns, Python keeps that variable alive in a closure attached to the multiplier function. 
# So, every time you call double(5), it uses that remembered factor.
def make_multiplier(factor):
    def multiplier(x):
        return x * factor
    return multiplier

double = make_multiplier(2)
triple = make_multiplier(3)

print(double(5))  
print(triple(5))   

10
15


In [11]:
def make_adder(n):
    def add(x):
        return x + n
    return add

add5 = make_adder(5)
print(add5(2))

7


## 2. Building a Simple Decorator

A decorator is a function that:
- takes a function as an argument
- defines a **wrapper** function inside that adds new behavior
- returns the wrapper

In [ ]:
def simple_decorator(func):
    def wrapper():
        print("Before calling the function.")
        func()
        print("After calling the function.")
    return wrapper

# Manual decoration
def greet():
    print("Hello!")
#function
simple_decorator(greet)()

Before calling the function.
Hello!
After calling the function.


In [1]:
def log(func):
    def wrapper():
        print("logging")
        func()
    return wrapper


In [2]:
def debug():
    print("Debug!")

def error():
    print("Error")

In [3]:
debug = log(debug)
debug()

logging
Debug!


In [4]:
error = log(error)
error()

logging
Error


## 3. The `@` Syntactic Sugar

Using `@decorator_name` above a function definition is exactly the same as `func = decorator(func)`. It's cleaner and more readable.

In [6]:
@simple_decorator
def say_hi():
    print("Hi there!")

say_hi()

Before calling the function.
Hi there!
After calling the function.


## 4. Decorators with Arbitrary Arguments (`*args`, `**kwargs`)

To make a decorator work with **any** function (with any number of positional or keyword arguments), use `*args` and `**kwargs` inside the wrapper.

In [5]:
# The key idea is that after decoration, calling add() no longer calls the original function directly—it calls wrapper(), and wrapper() decides when (and whether) 
# to invoke the original add() function that it remembers through a closure.
def universal_decorator(func):
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__} with args={args}, kwargs={kwargs}")
        result = func(*args, **kwargs)
        print(f"{func.__name__} returned {result}")
        return result
    return wrapper

Using

def wrapper(*args, **kwargs):

# makes the decorator universal—it can wrap almost
# any function regardless of how many positional or keyword arguments it accepts.

@universal_decorator
def add(a, b,c=5):
    return a + b + c

@universal_decorator
def multiply(x, y, z=1):
    return x * y * z

print(add(3, 5))
add(3,5,4)

Calling add with args=(3, 5), kwargs={}
add returned 13
13
Calling add with args=(3, 5, 4), kwargs={}
add returned 12


12

In [4]:
multiply(2, 4, z=10)

Calling multiply with args=(2, 4), kwargs={'z': 10}
multiply returned 80


80

## 5. Decorators That Take Arguments

Sometimes a decorator itself needs parameters (e.g., number of repetitions, log level).
This adds **one more level** of nesting: a function that returns a decorator.

In [9]:
# Whenever you see:

# @repeat(times=5)

# Mentally replace it with:

# greet = repeat(5)(greet)

# Now break it into two steps:

# temp = repeat(5)      # returns decorator_repeat
# greet = temp(greet)   # returns wrapper

# Then later:

# greet("Student")

# actually becomes:

# wrapper("Student")

# This mental expansion is the easiest way to understand decorators that take arguments.
def repeat(times):
    def decorator_repeat(func):
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator_repeat

@repeat(times=5)
def greet(name):
    print(f"Hello, {name}!")

greet("Student")

Hello, Student!
Hello, Student!
Hello, Student!
Hello, Student!
Hello, Student!


## 6. Chaining Multiple Decorators

You can stack decorators. They are applied **bottom‑up** (nearest to the function first, then upward).

In [27]:
def uppercase(func):
    def wrapper(namaste):
        result = func(namaste)
        return result.upper()
    return wrapper

def exclaim(func):
    def wrapper(msg):
        result = func(msg)
        return result + "!!!"
    return wrapper

@exclaim
@uppercase
def message(namaste):
    return "hello" + namaste

print(message("namaste"))   # HELLO!!!   (uppercase runs first, then exclaim)
# Multiple Decorators Execution
#
# @A
# @B
# def func():
#
# is equivalent to:
#
# func = A(B(func))
#
# Decorators are APPLIED from bottom to top.
#
# Execution flow when calling the function:
# 1. The outermost decorator's wrapper runs first.
# 2. It calls the next decorator's wrapper.
# 3. Eventually the original function executes.
# 4. Control returns back through each wrapper in reverse order.
#
# Example:
#
# @exclaim
# @uppercase
# def message():
#
# becomes:
#
# message = exclaim(uppercase(message))
#
# Call flow:
# message()
#   ↓
# exclaim.wrapper()
#   ↓
# uppercase.wrapper()
#   ↓
# original message()
#   ↓
# uppercase modifies result
#   ↓
# exclaim modifies result
#   ↓
# Final Output

HELLONAMASTE!!!


In [6]:
def uppercase(func):
    def wrapper():
        result = func()
        return result.upper()
    return wrapper

def exclaim(func):
    def wrapper():
        result = func()
        return result + "!!!"
    return wrapper

@exclaim
@uppercase
def message():
    return "hello"

print(message())   # HELLO!!!   (uppercase runs first, then exclaim)

HELLO!!!


In [21]:
message = exclaim(uppercase(message))
print(message())

TypeError: uppercase.<locals>.wrapper() missing 1 required positional argument: 'namaste'

In [ ]:
exclaim.wrapper()
    ↓
calls uppercase.wrapper()
    ↓
calls original message()
    ↓
returns "hello"
    ↓
uppercase makes it "HELLO"
    ↓
exclaim adds "!!!"
    ↓
final result = "HELLO!!!"

## 7. Real‑World Use Cases

### a) Timing a function (performance monitoring)

In [10]:
import time

def timer(func):
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        end = time.perf_counter()
        print(f"{func.__name__} took {end - start:.6f} seconds")
        return result
    return wrapper

@timer
def slow_sum(n):
    total = 0
    for i in range(n):
        total += i
    return total

slow_sum(10_00_000)

slow_sum took 0.100786 seconds


499999500000

In [9]:
import time

start = time.perf_counter()

for i in range(1000000):
    pass

end = time.perf_counter()

print(f"Elapsed time: {end - start} seconds")

Elapsed time: 0.05908439995255321 seconds


### b) Logging function calls (debugging / auditing)

In [35]:
def log(func):
    def wrapper(*args, **kwargs):
        print(f"[LOG] Calling {func.__name__}")
        result = func(*args, **kwargs)
        print(f"[LOG] {func.__name__} finished")
        return result
    return wrapper

@log
def divide(a, b):
    return a / b
    
@log
def multiply(a,b):
    return a*b
    
divide(10, 2)

[LOG] Calling divide
[LOG] divide finished


5.0

In [36]:
multiply(20, 5)

[LOG] Calling multiply
[LOG] multiply finished


100

### c) Authorisation (simple role check)

In [12]:
def requires_role(role):
    def decorator(func):
        def wrapper(user, *args, **kwargs):
            if user.get("role") != role:
                raise PermissionError(f"User needs {role} role")
            return func(user, *args, **kwargs)
        return wrapper
    return decorator

@requires_role("admin")
def delete_user(user, user_id):
    print(f"User {user_id} deleted by {user['name']}")

admin_user = {"name": "Alice", "role": "admin"}
delete_user(admin_user, 42)

guest_user = {"name": "arpana", "role": "guest"}

try:
    delete_user(guest_user, 42)
except PermissionError as e:
    print(e) # raises PermissionError
# Decorator with Arguments (Authorization Example)
#
# @requires_role("admin")
# def delete_user(...):
#
# Internally:
# delete_user = requires_role("admin")(delete_user)
#
# Execution:
# 1. requires_role("admin") stores the required role.
# 2. decorator(func) receives the original function.
# 3. wrapper() checks the user's role before calling the function.
# 4. If the role matches, the original function executes.
# 5. Otherwise, PermissionError is raised and the original function is not called.
#
# This pattern is commonly used for authentication and authorization.

User 42 deleted by Alice
User needs admin role


In [15]:
admin_user = {"name": "Alice", "role": "admin"}
delete_user(admin_user, 42)

User 42 deleted by Alice


## 8. Preserving Metadata with `functools.wraps`

Decorators replace the original function with the wrapper, which changes `__name__`, `__doc__`, etc. Use `@wraps` from `functools` to copy metadata.

In [38]:
def deco(func):
    def wrapper():
        return func()
    return wrapper

@deco
def hello():
    """Original doc"""
    pass

print(hello.__name__)
print(hello.__doc__)

wrapper
None


In [39]:
def deco(func):
    def wrapper():
        """Original doc1"""
        return func()
    return wrapper

@deco
def hello():
    """Original doc"""
    pass

print(hello.__name__)
print(hello.__doc__)

wrapper
Original doc1


In [1]:
from functools import wraps

def preserve_info_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        """Wrapper docstring"""
        return func(*args, **kwargs)
    return wrapper

@preserve_info_decorator
def important_function():
    """This function does something crucial."""
    pass

print(important_function.__name__)   
print(important_function.__doc__)    
# functools.wraps preserves the original function's metadata.
#
# Without @wraps:
#   __name__ -> wrapper
#   __doc__  -> wrapper's docstring
#
# With @wraps:
#   __name__ -> original function name
#   __doc__  -> original function docstring
#
# @wraps(func) copies metadata from the original function to the wrapper,
# making decorators transparent to users and development tools.
# Q: Why do we use functools.wraps in decorators?

# Answer: To preserve the original function's metadata (such as __name__, __doc__, and other attributes) after decoration.

important_function
This function does something crucial.


## 9. Class‑Based Decorators

Decorators can also be implemented as classes using the `__call__` method. Useful when you need to maintain state.

In [ ]:
#It makes an object behave like a function.

In [ ]:
# What __call__ Does

# This method runs whenever the object is called using parentheses ()

In [40]:
class CountCalls:
    def __init__(self, func):
        self.func = func
        self.count = 0

    def __call__(self, *args, **kwargs):
        self.count += 1
        print(f"Call {self.count} of {self.func.__name__}")
        return self.func(*args, **kwargs)

@CountCalls
def say_hello(name):
    print(f"Hello, {name}!")

say_hello("Alice")
say_hello("Bob")
print(f"Total calls: {say_hello.count}")

Call 1 of say_hello
Hello, Alice!
Call 2 of say_hello
Hello, Bob!
Total calls: 2


In [31]:
say_hello = CountCalls(say_hello)


## 10. Practice Exercise

**Task:** Write a decorator `retry(max_attempts, delay)` that:
- Tries to call the decorated function up to `max_attempts` times.
- If an exception occurs, waits `delay` seconds and retries.
- If all attempts fail, raises the last exception.

Use it to decorate a function that randomly fails (e.g., `import random; if random.random() < 0.7: raise ValueError`).

In [ ]:
import time
import random

def retry(max_attempts, delay=1):
    def decorator_retry(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"Attempt {attempt} failed: {e}")
                    if attempt == max_attempts:
                        raise
                    time.sleep(delay)
            return None
        return wrapper
    return decorator_retry

@retry(max_attempts=3, delay=0.5)
def unstable_function():
    if random.random() < 0.6:
        raise ValueError("Random failure!")
    return "Success!"

result = unstable_function()
print(result)
# A retry decorator automatically re-executes a function when it raises an exception, up to a specified number of attempts, with an optional delay between retries. It is commonly used for
# temporary failures such as network requests, API calls, or database connections.
# Retry Decorator
#
# @retry(max_attempts=3, delay=0.5)
#
# Internally:
# func = retry(3, 0.5)(func)
#
# - retry() stores the retry settings.
# - decorator_retry() receives the original function.
# - wrapper() executes the function.
# - If the function raises an exception:
#     • Print the error.
#     • Wait for 'delay' seconds.
#     • Retry until max_attempts.
# - If all attempts fail, 'raise' re-throws the last exception.
#
# Common use cases:
# • API requests
# • Database connections
# • Network operations
# • Temporary failures

## Summary

- **Decorators** modify or enhance functions without changing their source code.
- The `@` syntax is sugar for `func = decorator(func)`.
- Use `*args, **kwargs` in wrappers to make decorators universal.
- Decorators can accept **arguments** (three levels of nesting).
- Chain multiple decorators – order matters (bottom‑up execution).
- Real‑world uses: timing, logging, caching, authorisation, retries.
- Preserve metadata with `@wraps` from `functools`.
- Class‑based decorators are useful when state is needed.

Mastering decorators greatly improves code elegance, reusability, and separation of concerns.

In [ ]:
@classmethod
@staticmethod
@property

| Decorator       | Type of Method       | Description                                                                 |
|-----------------|-------------------|-----------------------------------------------------------------------------|
| `@classmethod`  | Class method       | Receives the class (`cls`) as first argument. Useful for factory methods or altering class state. |
| `@staticmethod` | Static method      | Does not receive `self` or `cls`. Acts like a regular function inside the class. |
| `@property`     | Instance property  | Turns a method into a read-only attribute. Access like `obj.attr` instead of `obj.method()`. |

In [41]:
# Example of @classmethod
class A:
    count = 0

    @classmethod
    def inc_count(cls):
        cls.count += 1
        return cls.count

print("Classmethod:", A.inc_count())

Classmethod: 1


In [42]:
# Example of @staticmethod
class B:
    @staticmethod
    def greet(name):
        return f"Hello {name}"

print("Staticmethod:", B.greet("Alice"))  # Output: Hello Alice

Staticmethod: Hello Alice


In [43]:
# Example of @property
class C:
    def __init__(self, x):
        self._x = x

    @property
    def x(self):
        return self._x

c = C(5)
print("Property:", c.x)  # Output: 5

Property: 5


In [44]:
class Demo:
    x = 10

    def instance_method(self):
        print("Instance method")

    @classmethod
    def class_method(cls):
        print("Class method:", cls.x)

    @staticmethod
    def static_method(a, b):
        print("Static method")
        print(a, b)

d = Demo()

d.instance_method()     # Instance method
Demo.class_method()     # Class method: 10
Demo.static_method(2,6)    # Static method

Instance method
Class method: 10
Static method
2 6


| Type            | First Parameter | Accesses Instance? | Accesses Class?                     | Called Using                       |
| --------------- | --------------- | ------------------ | ----------------------------------- | ---------------------------------- |
| Instance Method | `self`          | ✅ Yes              | ✅ Yes                               | `obj.method()`                     |
| Class Method    | `cls`           | ❌ No               | ✅ Yes                       | `Class.method()` or`obj.method()` |
| Static Method   | None            | ❌ No               | ❌ No (unless explicitly referenced) | `Class.method()`                   |

In [4]:
# PRACTICE
class Test:
    x = 0
    def __init__(self, name):
        self.name = name
    def take_x(self):
        self.x += 1
        return self.x
t = Test("testing")
print(t.take_x())

1
